# Using all five sensors

[Notebook 3](./3-subscribing-to-tof.ipynb) subscribed to the downward sensor, the one that runs
  by default. The Duckiedrone carries four more, pointing up, forward, left, and right, and they
  stay off until started.

This notebook starts them, checks that their topics appear at the rates [Notebook
  2](./2-duckiedrone-tof-architecture.ipynb) configures, and points the node from Notebook 3 at
  all five. That takes one new function and one changed line in the launch file.

By the end of the notebook you should be able to answer:

- What has to be running on the Duckiedrone before a topic exists to subscribe to?

- Where do the five topic names come from, and why can they be generated rather than listed?

- Why do the five readings in one printed line come from different measurement times?

## The five sensors

The five sensors, as [Notebook 2](./2-duckiedrone-tof-architecture.ipynb#step-1) described them:

| Sensor | Points | Configured rate (Hz) | Running at boot |
|---|---|---|---|
| `bottom` | Straight down | 30 | Yes |
| `front` | Forward | 5 | No |
| `left` | To the Duckiedrone's left | 5 | No |
| `right` | To the Duckiedrone's right | 5 | No |
| `top` | Straight up | 5 | No |

Left and right are named from the Duckiedrone's point of view, and its front is the side the
  camera faces.

Each sensor runs two containers, a [driver](./2-duckiedrone-tof-architecture.ipynb#step-2) and a
  [ROS 2 bridge](./2-duckiedrone-tof-architecture.ipynb#step-3), so starting the other four adds
  eight containers. The processor time those cost the Raspberry Pi is why the four stay off by
  default.

## Activity 1: start the other four sensors

<a id="activity-1"></a>

The eight containers belong to a **stack**, a group of containers started and stopped together,
  called `duckiedrone/extra_sensors`. Start it from the base station, the computer with the
  Duckietown Shell installed, not from the Duckiedrone:

```bash
dts stack up -H <robot> -d duckiedrone/extra_sensors
```

`-H` names the Duckiedrone and `-d` detaches, so the command returns instead of streaming the
  logs of eight containers.

Then check what changed, from a shell on the Duckiedrone opened as in [Step 1 of Notebook
  3](./3-subscribing-to-tof.ipynb#step-1).

**The containers:**

```bash
docker ps --format '{{.Names}}' | grep tof
```

Two names before, ten now: a `driver-tof-<sensor>` and a `ros2-tof-<sensor>` for each of the
  five sensors.

**The topics:** `ros2` is not installed on the Duckiedrone itself, so first open a shell inside
  a bridge, as in [Step 3 of Notebook 3](./3-subscribing-to-tof.ipynb#step-3):

```bash
docker exec -it ros2-tof-bottom bash
ros2 topic list | grep tof
```

```
/<robot>/bottom_tof_driver_node/range
/<robot>/front_tof_driver_node/range
/<robot>/left_tof_driver_node/range
/<robot>/right_tof_driver_node/range
/<robot>/top_tof_driver_node/range
```

Five topics where Notebook 3 found one, all named the same way. The next activity builds these
  names in code instead of typing them.

**The rates:** compare the sensor that always runs with one that was just started.

```bash
ros2 topic hz /<robot>/bottom_tof_driver_node/range
```

```
average rate: 30.008
	min: 0.018s max: 0.050s std dev: 0.00404s window: 245
```

```bash
ros2 topic hz /<robot>/front_tof_driver_node/range
```

```
average rate: 5.013
	min: 0.177s max: 0.216s std dev: 0.00688s window: 43
```

Both outputs come from a physical Duckiedrone and match the configured 30 Hz and 5 Hz. A virtual
  Duckiedrone can differ, so measure it rather than assume. Ctrl-C stops the command.

When done with this notebook, stop the stack to give the processor time back:

```bash
dts stack down -H <robot> duckiedrone/extra_sensors
```

## Activity 2: subscribe to all five

<a id="activity-2"></a>

The node takes a list of sensor names and builds each topic with `topics_for`, the function left
  to write.

Open `packages/solution/solution/tof_math.py` and fill in `topics_for` at its `TO IMPLEMENT`
  block. Every bridge names its topic the same way, so a sensor's topic follows from its name:
  the pattern is the one in the `ros2 topic list` output above, without the `/<robot>` prefix,
  which the node adds itself. One list comprehension is enough.

`ToFSubscriber` already calls `topics_for` on its sensor names, so once the function is written,
  the sensor names are all the node needs.

### Tell the launch file which sensors to use

The node receives its sensor list from the launch file in `packages/tof_listener/launch/`, which
  `dts code workbench` runs when it starts ([Activity 2 of Notebook
  3](./3-subscribing-to-tof.ipynb#activity-2)). The list is therefore set there.

Open `packages/tof_listener/launch/tof_listener_node.launch.py` and find the argument that names
  the sensors:

```python
sensor_names_arg = DeclareLaunchArgument(
    'sensor_names',
    default_value="['bottom']",
    description='Names of the ToF sensors to subscribe to.',
)
```

`default_value` is the one line to change. Name all five, in any order:

```python
    default_value="['bottom','front','left','right','top']",
```

Nothing else changes: the launch file never mentions topics, because `topics_for` builds them.
  Setting the line back to `['bottom']` restores the node from Notebook 3.

### Deploy it

The same two commands as [Notebook 3](./3-subscribing-to-tof.ipynb#activity-2):

```bash
dts code build -R <robot>
dts code workbench -R <robot>
```

The node now logs five subscriptions at startup, then a line a second with five numbers in it:

```
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/bottom_tof_driver_node/range  (sensor: bottom)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/front_tof_driver_node/range  (sensor: front)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/left_tof_driver_node/range  (sensor: left)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/right_tof_driver_node/range  (sensor: right)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/top_tof_driver_node/range  (sensor: top)
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.013m front=0.116m left=1.288m right=0.483m top=1.660m]
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.012m front=0.116m left=1.309m right=0.482m top=1.663m]
```

Check the startup lines first: each topic should be one of the five from [Activity
  1](#activity-1), with a publisher in `ros2 topic info`. If the extra sensors stack is down,
  the four extra subscriptions still succeed but receive nothing, and those sensors are missing
  from the printed line.

Then read the numbers. Those two lines come from a physical Duckiedrone resting on its legs, one
  second apart. `bottom` is below the sensor's minimum, as in [Activity 3 of Notebook
  3](./3-subscribing-to-tof.ipynb#activity-3), and `front`, `left`, and `right` are whatever
  surrounds the Duckiedrone.

### Five readings, five different times

The printed line does not show the rate difference. Each entry is the newest message from its
  sensor, and even a 5 Hz sensor delivers about five messages per printed line, so every entry
  can change from line to line. The rates are measured with `ros2 topic hz`, as in [Activity
  1](#activity-1).

What the line also hides is that its five numbers were measured at different times. When the
  line is printed, the `bottom` reading is at most about 33 ms old and a reading from any of the
  other four up to 200 ms old, plus the time each message took to arrive. At 1 m/s, a
  Duckiedrone moves 20 cm in 200 ms.

### What did not change

The code from Notebook 3 did not change. `_make_callback` gives each subscription a callback
  that carries its sensor's name, and `latest_readings` is keyed by that name, so the five
  sensors write to five separate entries.

## Going further

Each of these is optional, and each one builds on the node from [Activity 2](#activity-2):

- Subscribe to a subset: `default_value="['bottom','front']"` prints two entries. The node works
  with any number of names.

- Print how old each reading is. Every `ToFReading` carries `stamp_sec`, the time the driver
  read the measurement ([Notebook 2, Step 2](./2-duckiedrone-tof-architecture.ipynb#step-2)). In
  `cb_log`, subtract it from the current time, `self.get_clock().now().nanoseconds * 1e-9`, and
  print the result next to each range. Compare the age of `bottom` with the age of a sideways
  sensor.

- Watch one sideways sensor on its own while something moves in front of it: `ros2 topic echo
  /<robot>/front_tof_driver_node/range --field range`.

- Change a sideways sensor's rate in `/data/config/node/tof_driver_left/<robot>.yaml`, which on
  a physical Duckiedrone belongs to root, so edit it with `sudo`. To slow it down, lower
  `frequency`. Raising it above 5 Hz needs a shorter `timing_budget` as well: `frequency: 10`
  with `timing_budget: 100` is a valid combination ([Notebook
  2](./2-duckiedrone-tof-architecture.ipynb)). Restart `driver-tof-left` and then
  `ros2-tof-left`, in that order, and confirm the new rate with `ros2 topic hz`.